# Media aprovaçao por tema

In [ ]:
import pandas as pd

if not conn.is_connected():
    conn.reconnect()

cursor = conn.cursor()

try:
    # 1. Buscamos a relação Proposição <-> Tema e o Status da proposição
    # Usamos JOIN para saber o status de cada proposição vinculada a cada tema
    query = """
    SELECT pt.id_tema, p.status
    FROM proposicao_tema pt
    JOIN proposicoes p ON pt.id_proposicao = p.cd_proposicoes
    """

    cursor.execute(query)
    df_p = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

    # 2. Lógica de cálculo
    # Mantive o seu critério de sucesso
    status_sucesso = 'Transformado em Norma Jurídica'
    df_p['aprovada'] = df_p['status'] == status_sucesso

    # Agrupar pelo ID do tema e tirar a média
    stats = df_p.groupby('id_tema')['aprovada'].mean() * 100
    stats = stats.reset_index()
    stats.columns = ['id_tema', 'media']

    print("Iniciando atualização na tabela top_temas...")

    # 3. Preparar os dados para o UPDATE
    # Ajustei para os nomes das colunas da tabela top_temas (cd_tp_temas e media_aprovacao)
    dados_update = [
        (float(row['media']), int(row['id_tema']))
        for _, row in stats.iterrows() if pd.notna(row['id_tema'])
    ]

    # SQL de atualização
    sql_update = "UPDATE top_temas SET media_aprovacao = %s WHERE cd_tp_temas = %s"

    cursor.executemany(sql_update, dados_update)
    conn.commit()

    print(f"✅ Sucesso! {len(dados_update)} temas processados e atualizados na top_temas.")

except Exception as e:
    print(f"❌ Erro: {e}")
    conn.rollback()
finally:
    cursor.close()


Iniciando atualização na tabela top_temas...
✅ Sucesso! 32 temas processados e atualizados na top_temas.
